# Standalone HEP: Эволюция Признаков Высокого Порядка

**Hypergraph-Evolved Pipelines (HEP)** — это инновационный подход к автоматическому конструированию признаков (AutoFE), основанный на представлении генома как **динамического гиперграфа**. В отличие от классических методов, которые ограничены линейными или простыми полиномиальными комбинациями, HEP способен обнаруживать и синтезировать сложные нелинейные зависимости в данных.

### Ключевые преимущества HEP:
- **Интерпретируемость:** Каждое ребро гиперграфа — это конкретная математическая операция.
- **Эффективность поиска:** Эволюционный механизм фокусно исследует наиболее перспективные комбинации признаков.
- **Гибкость:** Возможность ограничения набора функций (max, min, std и др.) в зависимости от специфики задачи.

In [ ]:
import sys, os, inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.preprocessing import PolynomialFeatures

# 1. Настройка путей (КРИТИЧНО для импорта hep_engine)
current_path = os.getcwd()
project_root = os.path.abspath(os.path.join(current_path, '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 2. Настройка авто-перезагрузки
%load_ext autoreload
%autoreload 2

# 3. Импорт компонентов
try:
    from hep_engine.evaluator import FitnessEvaluator
    from hep_engine.optimizer import EvolutionaryOptimizer
    from hep_engine.visualizer import HEPVisualizer
    from hep_engine.core import Hypergraph
    print("✅ Библиотека hep_engine успешно загружена.")
    print(f"   Путь: {inspect.getfile(FitnessEvaluator)}")
    print(f"   Аргументы __init__: {inspect.signature(FitnessEvaluator.__init__)}")
except ImportError as e:
    print(f"❌ Ошибка импорта: {e}")
    print(f"   Текущий sys.path: {sys.path[:3]}...")

%matplotlib inline

## 2. Генерация "Невозможного" Датасета

Для демонстрации силы HEP мы создадим синтетический датасет, в котором целевая переменная $y$ зависит от базовых признаков через функции, которые сложно аппроксимировать стандартным моделям (например, случайному лесу без глубокого тюнинга):

1. **Синусоидальное произведение:** `sin(X0 * X1 * X2)` — требует понимания взаимодействия трех переменных.
2. **Скользящая вариация:** `std(X3:X7)` — агрегативный признак, скрывающий статистическую характеристику группы.
3. **Максимизация:** `max(X0:X1)` — нелинейный выбор.

In [ ]:
def generate_showcase_data(n_samples=1000, n_features=12):
    np.random.seed(42)
    X = np.random.uniform(-2, 2, (n_samples, n_features))
    
    # Скрытые нелинейные связи
    term1 = np.sin(X[:, 0] * X[:, 1] * X[:, 2])
    term2 = np.std(X[:, 3:7], axis=1)**2
    term3 = np.max(X[:, 0:1], axis=1)
    
    y = 10*term1 + 5 * term2 + term3 + np.random.normal(0, 0.1, n_samples)
    feature_names = [f"feat_{i}" for i in range(n_features)]
    return X, y, feature_names

X, y, feature_names = generate_showcase_data(n_samples=1000)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Данные сгенерированы: {X.shape[0]} примеров, {X.shape[1]} базовых признаков.")

## 3. Сравнение Baseline

Прежде чем запускать эволюцию, оценим качество стандартных подходов:
- **Raw RF:** Обучение на исходных признаках. Модель вынуждена сама строить дерево решений, пытаясь уловить нелинейности.
- **Polynomial Features (d=2):** Классическое расширение признакового пространства до попарных произведений. Это приводит к быстрому росту размерности данных, но не гарантирует нахождения сложных функций (например, `std`).

In [ ]:
# Эталонная модель для всех тестов
ref_model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)

# 1. RF Raw
rf_raw = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42).fit(X_train, y_train)
r2_raw = r2_score(y_test, rf_raw.predict(X_test))

# 2. RF + Polynomial Features (d=2)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly_train = poly.fit_transform(X_train)
X_poly_test = poly.transform(X_test)
rf_poly = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42).fit(X_poly_train, y_train)
r2_poly = r2_score(y_test, rf_poly.predict(X_poly_test))

print(f"R2 Score (RF Raw): {r2_raw:.4f}")
print(f"R2 Score (Polynomial d2): {r2_poly:.4f}")

## 4. Эволюционный HEP (Честное сравнение)

Теперь применим **EvolutionaryOptimizer**. Основная идея заключается в том, что гиперграф эволюционирует, меняя связи между признаками и применяя к ним агрегационные функции. 

**Настройки эксперимента:**
- **Размер популяции:** 10 особей.
- **Кроссовер и Мутация:** Высокие вероятности (0.5-0.6) для активного исследования пространства.
- **Элитизм:** Сохранение лучших 5 решений для стабильности.
- **Набор функций:** `['max', 'min', 'std', 'prod']`.

In [ ]:
# Создаем эвалюатор с подключаемой моделью
evaluator = FitnessEvaluator(X_train, y_train, model=ref_model, n_jobs=2)

optimizer = EvolutionaryOptimizer(
    pop_size=10, 
    mut_rate=0.5, 
    cross_rate=0.6, 
    elitism_count=5,
    available_functions=['max', 'min', 'std', 'prod']
)

print("Запуск эволюции гиперграфа...")
pop = optimizer.run(evaluator, n_generations=10, timeout=100000)
best_ind = pop.best()

print(f"\nЛучший фитнес (CV R2): {best_ind.fitness:.4f}")

## 5. Оценка и Визуализация

После завершения эволюции мы берем лучшую особь (гиперграф) и используем её как **Feature Transformer**. Новые признаки добавляются к исходным данным, и финальная модель (Random Forest) обучается на этом расширенном наборе.

In [ ]:
# Финальная оценка на отложенной выборке
X_hep_train = best_ind.genome.transform(X_train, cache=evaluator.cache)
X_hep_test = best_ind.genome.transform(X_test)

X_aug_train = np.hstack([X_train, X_hep_train])
X_aug_test = np.hstack([X_test, X_hep_test])

final_rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42).fit(X_aug_train, y_train)
r2_hep = r2_score(y_test, final_rf.predict(X_aug_test))

print(f"Итоговый R2 (HEP + RF): {r2_hep:.4f}")

# Сводный график
plt.figure(figsize=(10, 5))
results = [r2_raw, r2_poly, r2_hep]
labels = ['Raw', 'Poly d2', 'HEP (Our)']
colors = ['#cccccc', '#8888ff', '#22cc88']
plt.bar(labels, results, color=colors)
plt.ylim(0, 1.0)
for i, v in enumerate(results): plt.text(i, v+0.02, f"{v:.4f}", ha='center', fontweight='bold')
plt.title("Сравнение эффективности AutoFE методов")
plt.show()

In [ ]:
# Визуализация
viz = HEPVisualizer(output_dir='.')
print("Визуализация архитектуры найденных признаков:")
viz.plot_individual(best_ind, n_features=len(feature_names), title="Best Evolved Feature Structure")

## 6. Символьная декодировка

In [ ]:
print("Математические формулы признаков:")
for i, edge in enumerate(best_ind.genome.edges.values()):
    nodes = [feature_names[idx] for idx in edge.node_indices]
    print(f"  [Edge {i}]: {edge.function_name}({', '.join(nodes)})")